# 25. Canonicalization Test (Pipeline 06)

- Goal: inspect optional candidate-evidence coordinate families after normalization.
- Docs: `docs_eng/pipeline/06_canonicalization.md` / `docs/pipeline/06_canonicalization.md`
- Inputs: Normalized pose dataframe and canonicalization config from prior-stage cells.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Candidate availability, confidence, burden/residual, and prior-status summaries.


In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go

from movement.io import load_pose_csv
from movement.config import LANDMARKS, CONNECTIONS
from movement.canonicalization import (
    CanonicalizationConfig,
    MovementPlaneAlignmentConfig,
    ProtocolHeightLateralWidthAlignmentConfig,
    apply_canonicalization,
)
from movement.floor_reference import FloorReferenceConfig
from movement.normalization import (
    normalize_pose_by_hip_torso,
    check_normalization_result,
)
from movement.visualization import (
    create_pose_animation,
    create_pose_comparison_animation,
)

from movement.pipeline import (
    ExerciseDefinitionConfig,
    NormalizationConfig,
    PipelineConfig,
    ValidationConfig,
    run_pipeline,
)


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"

df = load_pose_csv(csv_path)

def estimate_frame_duration_ms(dataframe, default_ms=33):
    if "timestamp" not in dataframe.columns:
        return default_ms
    dt = dataframe["timestamp"].astype(float).diff().dropna()
    if dt.empty:
        return default_ms
    median_dt = float(dt.median())
    if median_dt <= 0:
        return default_ms
    return max(1, int(round(median_dt * 1000)))

frame_duration_ms = estimate_frame_duration_ms(df)
print(f"playback frame duration: {frame_duration_ms} ms (~{1000 / frame_duration_ms:.1f} fps)")

recording_view_camera = dict(
    eye=dict(x=0.0, y=-2.5, z=0.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
    projection=dict(type='orthographic'),
)

def apply_recording_view_camera(fig):
    fig.update_layout(scene_camera=recording_view_camera)
    return fig

df.head()


In [ ]:
norm_df, norm_report = normalize_pose_by_hip_torso(
    df=df,
    landmarks=LANDMARKS,
)

print(json.dumps(norm_report, indent=2, ensure_ascii=False))


## Direct Canonicalization Test

Canonicalization starts from the normalized coordinate family and creates separate `canon` columns. In this pass the output is candidate evidence: downstream stages continue to use `norm` coordinates, and score gravity is deferred to scoring policy.


In [ ]:
support_config = FloorReferenceConfig(
    enabled=True,
    method='support_contact_plane',
    coordinate_mode='norm',
    vertical_axis='y',
    support_landmarks=[
        'left_heel',
        'right_heel',
        'left_foot_index',
        'right_foot_index',
    ],
    diagnostic_landmarks=[
        'left_heel',
        'right_heel',
        'left_foot_index',
        'right_foot_index',
    ],
    visibility_threshold=0.7,
    max_anchor_residual_torso=0.08,
    correction_transform='rigid_rotation',
    camera_pitch_deg=0.0,
    camera_roll_deg=0.0,
    correction_strength=1.0,
    max_correction_torso=0.25,
)

movement_config = MovementPlaneAlignmentConfig(
    enabled=True,
    method='principal_motion_plane',
    fit_landmarks=[
        'left_hip',
        'left_knee',
        'left_ankle',
        'right_hip',
        'right_knee',
        'right_ankle',
    ],
    minimum_visible_landmark_ratio=0.7,
    correction_strength=0.5,
    max_rotation_deg=20.0,
    preserve_out_of_plane_residual=True,
)

protocol_height_config = ProtocolHeightLateralWidthAlignmentConfig(
    enabled=True,
    observed_height_level='H2',
    recommended_height_level='H2',
    require_height_match=True,
    correction_strength=0.3,
    max_scale_change=0.20,
    max_correction_torso=0.15,
    min_depth_offset_torso=0.05,
    visibility_threshold=0.6,
)

canonical_config = CanonicalizationConfig(
    enabled=True,
    coordinate_mode='norm',
    output_prefix='canon',
    report_only=True,
    downstream_coordinate_mode='norm',
    support_plane_alignment=support_config,
    movement_plane_alignment=movement_config,
    protocol_height_lateral_width_alignment=protocol_height_config,
)

canon_df, canon_report = apply_canonicalization(
    df=norm_df,
    landmarks=LANDMARKS,
    config=canonical_config,
)

print(json.dumps(canon_report, indent=2, ensure_ascii=False))


## Check 1: Canonical Output Columns Present


In [ ]:
assert canonical_config.report_only is True
assert canonical_config.downstream_coordinate_mode == 'norm'
assert isinstance(canon_report['candidate_available'], bool)
assert canon_report['candidate_confidence'] in {'not_available', 'high', 'moderate', 'low', 'not_emitted'}

missing_raw_norm = []
missing_canon = []
for landmark in LANDMARKS:
    for axis in ['x', 'y', 'z']:
        for family in ['', '_norm']:
            col = f'{landmark}{family}_{axis}' if family else f'{landmark}_{axis}'
            if col not in canon_df.columns:
                missing_raw_norm.append(col)
        canon_col = f'{landmark}_canon_{axis}'
        if canon_col not in canon_df.columns:
            missing_canon.append(canon_col)
assert not missing_raw_norm, f'missing raw/norm columns: {missing_raw_norm[:8]}'

for col in [
    'canonicalization_valid',
    'canonicalization_candidate_available',
    'canonicalization_candidate_confidence',
    'canonicalization_burden_level',
    'canonicalization_confidence',
    'canonicalization_correction_abs_frame',
]:
    assert col in canon_df.columns

summary = pd.DataFrame([
    {
        'candidate_available': canon_report['candidate_available'],
        'candidate_confidence': canon_report['candidate_confidence'],
        'burden_level': canon_report['burden_level'],
        'missing_canon_columns': len(missing_canon),
        'report_only': canon_report['report_only'],
        'downstream_coordinate_mode': canon_report['downstream_coordinate_mode'],
    }
])
display(summary)
if missing_canon:
    print('canon columns not produced for first items:', missing_canon[:8])
print('PASS: raw/norm coordinate families and canonicalization review metadata are inspectable')


## Check 2: Candidate Summary

Use the compact candidate fields for routine review, then inspect the prior evidence table to see which priors were configured, available, and measurable. Prior counts are intentionally omitted because they hide which prior mattered.

In [ ]:
support_report = canon_report['prior_reports']['support_plane_alignment']
movement_report = canon_report['prior_reports']['movement_plane_alignment']
protocol_height_report = canon_report['prior_reports'][
    'protocol_height_lateral_width_alignment'
]
def prior_available(prior_report):
    return bool(prior_report and prior_report.get('status') in {'applied', 'warning'})


def prior_reason(prior_report):
    if not prior_report:
        return 'not_configured'
    if prior_available(prior_report):
        return 'available'
    notes = prior_report.get('confidence_notes', []) or []
    return notes[0] if notes else str(prior_report.get('status', 'not_available'))


def support_key_metric(prior_report):
    if not prior_report:
        return ''
    residual = (prior_report.get('anchor_residual_summary') or {}).get('max')
    return f"anchor_frames={prior_report.get('num_anchor_frames')}; residual_max={residual}"


def movement_key_metric(prior_report):
    if not prior_report:
        return ''
    residual = (prior_report.get('out_of_plane_residual_ratio_after') or {}).get('p90')
    return f"rotation_deg={prior_report.get('applied_rotation_deg')}; residual_p90={residual}"


def protocol_key_metric(prior_report):
    if not prior_report:
        return ''
    return (
        f"height={prior_report.get('observed_height_level')}->"
        f"{prior_report.get('recommended_height_level')}; "
        f"max_scale_delta={prior_report.get('max_scale_delta')}"
    )


candidate_summary = pd.DataFrame([
    {
        'candidate_available': canon_report['candidate_available'],
        'candidate_confidence': canon_report['candidate_confidence'],
        'burden_level': canon_report['burden_level'],
        'downstream_coordinate_mode': canon_report['downstream_coordinate_mode'],
        'max_correction_torso': canon_report['max_correction_torso'],
        'median_correction_torso': canon_report['median_correction_torso'],
    }
])

prior_evidence = pd.DataFrame([
    {
        'prior_id': 'support_plane_alignment',
        'configured_on': support_config.enabled,
        'candidate_available': prior_available(support_report),
        'reason': prior_reason(support_report),
        'key_metric': support_key_metric(support_report),
    },
    {
        'prior_id': 'movement_plane_alignment',
        'configured_on': movement_config.enabled,
        'candidate_available': prior_available(movement_report),
        'reason': prior_reason(movement_report),
        'key_metric': movement_key_metric(movement_report),
    },
    {
        'prior_id': 'protocol_height_lateral_width_alignment',
        'configured_on': protocol_height_config.enabled,
        'candidate_available': prior_available(protocol_height_report),
        'reason': prior_reason(protocol_height_report),
        'key_metric': protocol_key_metric(protocol_height_report),
    },
])

display(candidate_summary)
display(prior_evidence)


## Normalized Visualization

In [ ]:
fig_norm = create_pose_animation(
    df=norm_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode='norm',
    frame_duration=frame_duration_ms,
    height=700,
    width=950,
    show_text=False,
    title='p01 squat normalized coordinates',
)

apply_recording_view_camera(fig_norm)
fig_norm

## Canonical Visualization


In [ ]:
fig_canon = create_pose_animation(
    df=canon_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode='canon',
    frame_duration=frame_duration_ms,
    height=700,
    width=950,
    show_text=False,
    title='p01 squat canonical coordinates',
)

apply_recording_view_camera(fig_canon)
fig_canon


## Normalized vs Canonical Comparison

This is a p01 squat review gate. Blue is the default downstream `norm` coordinate
family; red is the candidate-evidence `canon` coordinate family when canonicalization can
produce it. Use the report table above when real-data availability leaves the
canonical candidate partial or skipped.


In [ ]:
fig_compare = create_pose_comparison_animation(
    df=canon_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_modes=('norm', 'canon'),
    names=('Normalized', 'Canonical'),
    frame_duration=frame_duration_ms,
    height=750,
    width=1000,
    show_text=False,
    title='p01 squat normalized vs canonical pose',
)

apply_recording_view_camera(fig_compare)
fig_compare


## Canonicalization Diagnostics

Support-plane height is a residual against the pseudo-floor prior. Movement-plane residual ratios describe how much motion remains outside the aligned review plane. These are diagnostic signals, not movement-quality deductions.


In [ ]:
fig_diag = go.Figure()

for landmark in support_config.diagnostic_landmarks:
    col = f'{landmark}_canon_support_plane_height'
    fig_diag.add_trace(
        go.Scatter(
            x=canon_df['frame'],
            y=canon_df[col],
            mode='lines',
            name=col,
        )
    )

fig_diag.add_trace(
    go.Scatter(
        x=canon_df['frame'],
        y=canon_df['canonicalization_correction_abs_frame'],
        mode='lines',
        name='canonicalization_correction_abs_frame',
        line=dict(dash='dash'),
    )
)

if 'canonicalization_lateral_width_scale_delta_frame' in canon_df.columns:
    fig_diag.add_trace(
        go.Scatter(
            x=canon_df['frame'],
            y=canon_df['canonicalization_lateral_width_scale_delta_frame'],
            mode='lines',
            name='lateral_width_scale_delta_frame',
            line=dict(dash='dot'),
        )
    )

fig_diag.update_layout(
    title='p01 squat canonicalization diagnostics',
    xaxis_title='Frame',
    yaxis_title='torso_length_ratio',
    height=420,
    width=950,
)

fig_diag


## Check 3: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation = ValidationConfig(enabled=True)
cfg.normalization = NormalizationConfig(enabled=True, keep_reference_columns=True)
cfg.canonicalization = canonical_config

pipe_df, pipe_report = run_pipeline(df, cfg, landmarks=LANDMARKS)

assert 'canonicalization' in pipe_report
assert isinstance(pipe_report['canonicalization']['candidate_available'], bool)
assert pipe_report['canonicalization']['candidate_confidence'] in {'not_available', 'high', 'moderate', 'low', 'not_emitted'}
assert pipe_report['canonicalization']['report_only'] is True
assert pipe_report['canonicalization']['downstream_coordinate_mode'] == 'norm'

print('PASS: canonicalization report is present; downstream mode remains norm')
print(json.dumps(pipe_report['canonicalization'], indent=2, ensure_ascii=False))


## Check Summary

This notebook cell is a compact execution/QC checkpoint. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.
